# Time Series Inventory Forecasting Results
This notebook visualizes the results of our ML forecasting model (XGBoost) vs actual sales and shows the business impact of our inventory optimization policy.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
xgb_results = pd.read_parquet('../data/xgboost_forecast_results.parquet')
prophet_results = pd.read_parquet('../data/prophet_forecast_results.parquet')
comparison = pd.read_csv('../data/model_comparison.csv')
inventory_results = pd.read_parquet('../data/inventory_optimization_results.parquet')

print("="*50)
print("FORECASTING MODEL COMPARISON")
print("="*50)
print(comparison.to_string(index=False))
print("\nConclusion: XGBoost achieved lower error metrics and is used for inventory optimization.")
print("="*50)

# Show business impact metrics
total_stockout_naive = inventory_results['stockout_units_naive'].sum()
total_stockout_ml = inventory_results['stockout_units_ml'].sum()
total_holding_cost = inventory_results['holding_cost'].sum()

reduction = (total_stockout_naive - total_stockout_ml) / total_stockout_naive * 100 if total_stockout_naive > 0 else 0

print("\n" + "="*50)
print("INVENTORY OPTIMIZATION RESULTS (SIMULATED)")
print("="*50)
print(f"Total Stockout Units (Naive ROP): {total_stockout_naive:,.0f}")
print(f"Total Stockout Units (ML ROP with Safety Stock): {total_stockout_ml:,.0f}")
print(f"Business Impact: Reduced projected stockout rate by {reduction:.1f}%")
print(f"Incremental Carrying Cost for Safety Stock: ${total_holding_cost:,.2f}")
print("="*50)

## Forecast vs Actual for Multiple SKUs (Prophet vs XGBoost)

In [ ]:
# Get a few sample SKUs
skus = xgb_results['item_id'].unique()[:4]

plt.figure(figsize=(15, 10))
for i, sku in enumerate(skus):
    plt.subplot(2, 2, i+1)
    sku_xgb = xgb_results[xgb_results['item_id'] == sku].sort_values('date')
    sku_prophet = prophet_results[prophet_results['item_id'] == sku].sort_values('date')
    
    plt.plot(sku_xgb['date'], sku_xgb['actual_sales'], label='Actual Sales', marker='o', alpha=0.6, color='black')
    plt.plot(sku_xgb['date'], sku_xgb['predicted_sales'], label='XGBoost Forecast', linestyle='-', linewidth=2, color='blue')
    plt.plot(sku_prophet['date'], sku_prophet['predicted_sales'], label='Prophet Forecast', linestyle='--', linewidth=2, color='orange')
    
    plt.title(f"Forecast Comparison: {sku}")
    plt.legend()
    plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('../data/forecast_vs_actual.png')
plt.show()